# PROYECTO SPRINT 10

In [65]:
# ----------- Importaciones

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

# 1. Abre y examina el archivo de datos. Dirección al archivo


In [66]:
"""el data set proporcionado se lee para poder trabajar con el df"""
df = pd.read_csv(r"C:\\Users\\sasor\\Desktop\\Tripleten\\Sprint 10\\Proyecto\\users_behavior.csv")

### Revisiones secundarias para verificar que los datos del df esten en orden

In [67]:
"""esto nos permite revisar que se haya cargado correctamente"""
print(df.head())

   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0


In [68]:
"""aqui se verifica el tipo de datos de cada una"""
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None


In [69]:
print(df.dtypes)

calls       float64
minutes     float64
messages    float64
mb_used     float64
is_ultra      int64
dtype: object


Como todo se encuentra en orden no es necesario hacer una limpieza del df, por lo que se va directamente a trabajar con el.

# 2. Segmenta los datos fuente en un conjunto de entrenamiento, uno de validación y uno de prueba.

In [70]:
# se divide todo en 3 partes con proporción 3:1:1 donde 60% es de entrenamiento, 20% de validación y 20%  a la prueba).
features = df.drop(["is_ultra"], axis=1)
target = df["is_ultra"]

features_train, features_rest, target_train, target_rest = train_test_split(features, target, test_size=0.4, random_state=12345)

features_valid, features_test, target_valid, target_test = train_test_split(features_rest, target_rest, test_size=0.5, random_state=12345)

In [71]:
"""Verificacion de que las proporciones de clase se mantuvieron"""
print("entrenamiento:", features_train.shape, target_train.shape)
print("validacion", features_valid.shape, target_valid.shape)
print("prueba", features_test.shape, target_test.shape)

entrenamiento: (1928, 4) (1928,)
validacion (643, 4) (643,)
prueba (643, 4) (643,)


# 3. Investiga la calidad de diferentes modelos cambiando los hiperparámetros. Describe brevemente los hallazgos del estudio.

In [72]:
"""arbol de decision"""

best_tree_score = 0
best_tree_depth = 0
for depth in range(1, 11):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(features_train, target_train)
    score = model.score(features_valid, target_valid)
    print(f"max_depth = {depth}: exactitud en validacion = {score:.4f}")
    if score > best_tree_score:
        best_tree_score = score
        best_tree_depth = depth
print(f"mdjor arbol de decision es max_depth={best_tree_depth},"f"exactitud en validacion={best_tree_score:.4f}")

max_depth = 1: exactitud en validacion = 0.7543
max_depth = 2: exactitud en validacion = 0.7823
max_depth = 3: exactitud en validacion = 0.7854
max_depth = 4: exactitud en validacion = 0.7792
max_depth = 5: exactitud en validacion = 0.7792
max_depth = 6: exactitud en validacion = 0.7838
max_depth = 7: exactitud en validacion = 0.7823
max_depth = 8: exactitud en validacion = 0.7792
max_depth = 9: exactitud en validacion = 0.7823
max_depth = 10: exactitud en validacion = 0.7745
mdjor arbol de decision es max_depth=3,exactitud en validacion=0.7854


El mejor resultado es aquel con max_deptch=3, pero a partir del 4 se estanca y baja. Esto quiere decir que el arbol esta con un patron de sobreajuste donde el entrenamiento si mejora, pero la validacion tiene un efecto donde cae o baja. Sin embargo este resultad supera el umbral minimo de 0.75

In [ ]:
"""bosque aleatorio"""

best_forest_score = 0
best_forest_est = 0
best_forest_depth = 0
for est in range(10, 101, 10):
    for depth in range(1, 11):
        model = RandomForestClassifier(
            random_state=12345, n_estimators=est, max_depth=depth)
        model.fit(features_train, target_train)
        score = model.score(features_valid, target_valid)
        if score > best_forest_score:
            best_forest_score = score
            best_forest_est = est
            best_forest_depth = depth
 
print(f"mejor bosque aleatorio -> n_estimators={best_forest_est},"f"max_depth={best_forest_depth},"f"exactitud en validacion={best_forest_score:.4f}")

Con n_estimators=40 y con max_depth=8 se puede ver que la exactitud supera incluso al arbol de decision. Eso significa que aguanta mas que el otro antes de sobreajustarse porque el promedio de muchos arboles son los que cancelan los errores individuales. Supera el umbral de exactitud de 0.75

In [ ]:
"""regresion logistica"""

log_model = LogisticRegression(random_state=12345, solver="liblinear")
log_model.fit(features_train, target_train)
log_score_train = log_model.score(features_train, target_train)
log_score_valid = log_model.score(features_valid, target_valid)
print(f"Exactitud en entrenamiento: {log_score_train:.4f}")

Este es el unico modelo donde se queda por debajo del umbral minimo de exactitud, comparado con los otros, este obtuvo el resultado mas bajo. Podria decirse que la relacion entre los MB usados junto con las demas columnas y el plan elegido no es una relacion lineal.

# 4. Comprueba la calidad del modelo usando el conjunto de prueba.


In [ ]:
final_model = RandomForestClassifier(
    random_state=12345, n_estimators=best_forest_est, max_depth=best_forest_depth)
final_model.fit(features_train, target_train)
 
test_predictions = final_model.predict(features_test)
test_accuracy = accuracy_score(target_test, test_predictions)

In [ ]:
"""evaluacion del modelo"""
print(f'modelo final: RandomForestClassifier(n_estimators={best_forest_est},'
      f'max_depth={best_forest_depth})')
print(f'exactitud:{test_accuracy:.4f}')

if test_accuracy >= 0.75:
    print("el modelo tiene un umbral igual o mayor de exactitud a 0.75")
else:
    print("El modelo tiene un umbral menor de exactitud")

Dado que el bosque aleatorio fue el mejor modelo de todos, el mas exacto, es el usado. Se elige precisamente por ser el mas preciso ya que los errores de los demas modelos se intentan "corregir" con este.
El peor, el mas bajo fue la regresion logistica. Por su naturlaeza no se puede escoger, como la relacion del plan del usuario y las caracteristicas del usuario no son lineales, por eso no sirve y se descarta.

# 5. Prueba de cordura


In [ ]:
dummy_frequent = DummyClassifier(strategy="most_frequent", random_state=12345)
dummy_frequent.fit(features_train, target_train)
dummy_frequent_acc = dummy_frequent.score(features_test, target_test)
print(f'dummy - exactitud en prueba:'f'{dummy_frequent_acc:.4f}')
 
dummy_stratified = DummyClassifier(strategy="stratified", random_state=12345)
dummy_stratified.fit(features_train, target_train)
dummy_stratified_acc = dummy_stratified.score(features_test, target_test)
print(f'dummy - exactitud en prueba: 'f'{dummy_stratified_acc:.4f}')
 
if test_accuracy > dummy_frequent_acc and test_accuracy > dummy_stratified_acc:
    print("modelo supera la prueba de cordura")
else:
    print("mal el modelo, revisar nuevamente.")

Como se vio en las clases, la prueba de cordura sirve para comparar el modelo con otro aleatorio, para verificar si realmente tiene sentido. El Dummyclassifier nos dice cual es la clase que mas aparece en el entrenamiento, en este caso se sabia de antemano que el 69% de usuarios tiene smart y la exactitud de esta prueba dio .6843 en la primera parte, es la proporcion del subconjunto en general.
pero ahora con stratified se predice que, del 69% de smart y 31% de ultra sacó 0.5465, aun mas bajo que antes.